# Webserv - Résumé du travail réalisé

## Point de départ — Analyse initiale

### Fichiers existants avant l'intégration

| Fichier | État |
|---------|------|
| `Server.hpp / Server.cpp` | Boucle `poll()`, accept, read/write, timeouts |
| `Client.hpp / Client.cpp` | `readData()`, `writeData()`, `reset()`, keep-alive |
| `ConfigParser.cpp` | Parse du fichier `.conf` → `std::vector<ServerConfig>` |
| `Request.hpp / Request.cpp` | Parsing HTTP (méthode, URI, headers, body) |
| `Response.hpp / Response.cpp` | Construction réponse (`setStatus`, `setHeader`, `setBody`, `build`) |
| `Router.hpp / Router.cpp` | Dispatch selon `RouteType` (fichier statique, redirect, erreur) |
| `CGIHandler.hpp / CGIHandler.cpp` | `executeCGI()` bloquant (fork + execve + waitpid) |
| `Dico.hpp` | Structures : `ServerConfig`, `LocationConfig`, `Request`, `Response`, `CGIData`, `ClientData`, `RouteResult` |

### Ce qui manquait

1. **Router** : pas intégré dans `Server.cpp` (réponse hardcodée)
2. **POST / DELETE** : non implémentés
3. **Autoindex** : non implémenté
4. **CGI non-bloquant** : `executeCGI()` bloquait toute la boucle `poll()`
5. **Validations** : `client_max_body_size`, multipart, virtual hosts

## Étape 1 — Intégration du Router dans Server.cpp

### Problème
Le serveur renvoyait une réponse hardcodée (`200 OK Hello World`) au lieu d'utiliser le Router.

### Solution
Remplacement dans `handleClientEvents()` du code hardcodé par :

```cpp
RouteResult result = Router::route(req, client->getServerPort(), _configs);

switch (result.type) {
    case ROUTE_FILE:      Router::serveFile(result.filepath, *res); break;
    case ROUTE_REDIRECT:  Router::handleRedirect(result, *res); break;
    case ROUTE_CGI:       /* executeCGI() */ break;
    case ROUTE_DIRECTORY:  /* 403 par défaut */ break;
    case ROUTE_UPLOAD:    /* POST upload */ break;
    case ROUTE_DELETE:    /* DELETE */ break;
    case ROUTE_ERROR:     Router::makeError(result.error_code, *res); break;
}
```

### Fichiers modifiés
- `srcs/core/Server.cpp` : dispatch via `Router::route()`

## Étape 2 — Implémentation POST (upload) et DELETE

### POST — Upload de fichiers

Ajout dans `Router.cpp` :
```cpp
bool handleUpload(const Request& request, const std::string& uploadDir, Response& response)
```
- Extraction du nom de fichier depuis le header `Content-Disposition` ou l'URI
- Écriture du body dans `uploadDir + filename`
- Réponse `201 Created`

### DELETE — Suppression de fichiers

```cpp
bool handleDelete(const std::string& filepath, Response& response)
```
- Vérifie que le fichier existe (`access()`)
- Supprime avec `remove()`
- Réponse `200 OK` ou `404 Not Found`

### Refactoring du Router

Le fichier `Router.cpp` devenait trop grand → découpage en fonctions par méthode :
- `routeGET()` : fichier statique, redirect, directory
- `routePOST()` : upload, CGI
- `routeDELETE()` : suppression

### Fichiers modifiés
- `srcs/http/Router.cpp` : ajout `handleUpload()`, `handleDelete()`, refactoring
- `includes/http/Router.hpp` : nouvelles déclarations

## Étape 3 — Autoindex (listing de répertoire)

### Principe
Quand `autoindex on` dans la config et qu'une URI pointe vers un répertoire sans `index.html` → génération d'une page HTML listant les fichiers.

### Implémentation

**Nouveau fichier** : `srcs/http/Autoindex.cpp`

```cpp
namespace Router {
bool handleAutoindex(const std::string& dirpath, const std::string& uri, Response& response)
{
    DIR* dir = opendir(dirpath.c_str());
    // Collecte des entrées avec readdir()
    // Tri alphabétique
    // Génération HTML : tableau avec nom, taille, date
    // Liens cliquables basés sur l'URI (pas le chemin filesystem)
    ResponseBuilder::setStatus(response, 200);
    ResponseBuilder::setHeader(response, "Content-Type", "text/html");
    ResponseBuilder::setBody(response, html.str());
    return true;
}
}
```

### Intégration dans Server.cpp
```cpp
case ROUTE_DIRECTORY:
    Router::handleAutoindex(result.filepath, req->uri, *res);
    break;
```

### Fichiers créés/modifiés
- `srcs/http/Autoindex.cpp` (**créé**)
- `includes/http/Router.hpp` : signature mise à jour
- `srcs/core/Server.cpp` : case `ROUTE_DIRECTORY`
- `Makefile` : ajout `http/Autoindex.cpp`

## Étape 4 — CGI non-bloquant

### Problème
`executeCGI()` utilisait `waitpid()` bloquant → le serveur entier était gelé pendant l'exécution d'un script CGI.

### Architecture de la solution

```
CLIENT_READING → parse requête
       ↓
ROUTE_CGI détecté → startCGI() → fork + execve
       ↓
CLIENT_WAITING_CGI → pipe_out ajouté à poll()
       ↓
poll() détecte POLLIN sur pipe → lecture buffer
       ↓
EOF/POLLHUP → finishCGI() → parse output → CLIENT_WRITING
```

### Nouveau fichier : `srcs/cgi/CGIAsync.cpp`

**`startCGI()`** : fork + execve, pipe_out en `O_NONBLOCK`, retourne `CGIData`

**`finishCGI()`** : close pipe, waitpid, parse output CGI, retourne `Response`

### Modifications dans Server.cpp

- **`buildPollFds()`** : ajout des pipes CGI après les sockets clients, `_cgiStartIndex` pour partitionner
- **`handleClientEvents()`** : boucle limitée à `_cgiStartIndex`, `ROUTE_CGI` appelle `startCGI()` au lieu de `executeCGI()`
- **`handleCGIEvents()`** : **nouveau** — itère à partir de `_cgiStartIndex`, lit les pipes, détecte EOF/POLLHUP, appelle `finishCGI()`
- **`checkTimeouts()`** : timeout CGI de 30s → `kill(SIGKILL)` + `waitpid()` + réponse 504
- **`run()`** : ajout de `handleCGIEvents()` dans la boucle principale

### Attributs ajoutés à Server
```cpp
std::map<int, int> _cgiPipeToClient;  // pipe_out fd → client fd
size_t _cgiStartIndex;                // partition dans _pollFds
```

### Fichiers créés/modifiés
- `srcs/cgi/CGIAsync.cpp` (**créé**)
- `includes/cgi/CGIHandler.hpp` : déclarations `startCGI()`, `finishCGI()`, fonctions utilitaires
- `srcs/cgi/CGIHandler.cpp` : suppression du `static` des fonctions partagées
- `includes/core/Server.hpp` : nouveaux attributs et méthode
- `srcs/core/Server.cpp` : refonte du dispatch CGI
- `includes/core/Client.hpp` / `srcs/core/Client.cpp` : getter `getCGIData()`
- `Makefile` : ajout `cgi/CGIAsync.cpp`

## Résultats des tests

| # | Test | Résultat |
|---|------|----------|
| 1 | GET fichier statique (`/index.html`) | 200 OK |
| 2 | GET fichier inexistant (`/nope`) | 404 Not Found |
| 3 | GET redirect (`/old-page`) | 301 Moved Permanently |
| 4 | POST upload (`/uploads/test.txt`) | 201 Created |
| 5 | GET fichier uploadé | 200 OK |
| 6 | DELETE fichier (`/uploads/test.txt`) | 200 OK |
| 7 | GET après DELETE | 404 Not Found |
| 8 | CGI Python (`/cgi-test/hello.py`) | 200 OK |
| 9 | CGI Bash (`/cgi-test/test.sh`) | 200 OK |
| 10 | Méthode non autorisée (PUT) | 405 Method Not Allowed |
| 11 | Autoindex ON (`/uploads/`) | 200 OK (listing HTML) |
| 12 | Autoindex OFF (`/cgi-test/`) | 403 Forbidden |
| 13 | CGI concurrent + requête statique | Statique: 7ms, CGI: 25-87ms (non-bloquant vérifié) |

## Arborescence des fichiers

```
Webserv/
├── includes/
│   ├── core/
│   │   ├── Server.hpp
│   │   └── Client.hpp
│   ├── http/
│   │   ├── Router.hpp
│   │   ├── Request.hpp
│   │   └── Response.hpp
│   ├── config/
│   │   └── ConfigParser.hpp
│   ├── cgi/
│   │   └── CGIHandler.hpp
│   └── utils/
│       └── Dico.hpp
├── srcs/
│   ├── main.cpp
│   ├── core/
│   │   ├── Server.cpp
│   │   └── Client.cpp
│   ├── http/
│   │   ├── Router.cpp
│   │   ├── Request.cpp
│   │   ├── Response.cpp
│   │   └── Autoindex.cpp      ← NOUVEAU
│   ├── config/
│   │   └── ConfigParser.cpp
│   └── cgi/
│       ├── CGIHandler.cpp
│       └── CGIAsync.cpp        ← NOUVEAU
├── config/
│   └── default.conf
├── Makefile
└── Docs/
    └── resume.ipynb
```

## Ce qu'il reste à faire

- [ ] Validation `client_max_body_size` → réponse **413 Request Entity Too Large**
- [ ] Parsing multipart `form-data` pour les uploads
- [ ] Virtual hosts : routing par `server_name` (Host header)
- [ ] Pages d'erreur personnalisées depuis la config
- [ ] Tests de stress / charge
- [ ] Conformité complète HTTP/1.1